In [ ]:
# Import pandas library to work with datasets.
import pandas as pd

In [ ]:
# Install openpyxl to enable reading Excels.
%pip install openpyxl


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Read the Excels and store them into variables.
gender_df_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_gender_gen_election.xlsx")
age_df_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_age_gen_election.xlsx")
race_df_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_race_gen_election.xlsx")
total_df_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_total_gen_election.xlsx")

In [ ]:
# Make copies of the datasets and store them into variables.
# We will primarily be working with the following variables in data validation, cleaning, etc.
gender_df = gender_df_og.copy()
age_df = age_df_og.copy()
race_df = race_df_og.copy()
total_df = total_df_og.copy()

In [ ]:
# Clean the headers by removing whitespace and linebreaks.
for df in [gender_df, age_df, race_df, total_df]:
    df.columns = df.columns.str.strip().str.replace("\n"," ")

# Rename the columns in the four datasets to avoid any same-name clashes when merging the datasets.
gender_df = gender_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Gender"})
age_df = age_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Age"})
race_df = race_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Race"})
total_df = total_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Total"})

In [ ]:
# Clean the County values.
for df in [gender_df, age_df, race_df, total_df]:
    df["County"] = df["County"].str.strip()
    df["County"] = df["County"].str.replace("\n"," ")
    df["County"] = df["County"].str.title()

In [ ]:
# Join the three demographics datasets.
merged = gender_df.merge(age_df, on="County").merge(race_df, on="County")

In [ ]:
# List the columns of the merged dataset to check that they were joined correctly - they were.
merged.columns.tolist()

['County',
 'Total Ballots Cast - Gender',
 'Total Female',
 'Total Male',
 'Total Not Identified',
 'Total Ballots Cast - Age',
 'Age 18-29',
 'Age 30-39',
 'Age 40-49',
 'Age 50-59',
 'Age 60-69',
 'Age 70-79',
 'Age 80-89',
 'Age 90-99',
 'Age 100+',
 'Total Ballots Cast - Race',
 'Total Asian (A)',
 'Total American Indian (AI)',
 'Total Black (B)',
 'Total Federally- Registered (may be of any race) (F)',
 'Total Hispanic (H)',
 'Total Korean (K)',
 'Total Other (O)',
 'Total White (W)',
 'Total Not Identified (U)']

In [ ]:
# If all three values in a row match, the result will be stored as True in the new column "equal". 
# Otherwise, it will be stored as "False".
merged["equal"] = (
    (merged["Total Ballots Cast - Gender"] == merged["Total Ballots Cast - Age"]) &
     (merged["Total Ballots Cast - Gender"] == merged["Total Ballots Cast - Race"])
)

# Gathers all of the records shown as "False" and stores them into "mismatches".
mismatches = merged[~merged["equal"]]

In [ ]:
print(mismatches)
# Output: Empty Dataframe.
# This means there are no mismatches in total ballots cast among the three datasets. 

Empty DataFrame
Columns: [County, Total Ballots Cast - Gender, Total Female, Total Male, Total Not Identified, Total Ballots Cast - Age, Age 18-29, Age 30-39, Age 40-49, Age 50-59, Age 60-69, Age 70-79, Age 80-89, Age 90-99, Age 100+, Total Ballots Cast - Race, Total Asian (A), Total American Indian (AI), Total Black (B), Total Federally- Registered (may be of any race) (F), Total Hispanic (H), Total Korean (K), Total Other (O), Total White (W), Total Not Identified (U), equal]
Index: []

[0 rows x 26 columns]


In [26]:
# We can now compare our merged dataset with total_df by joining the two.
compare = merged.merge(total_df, on="County")

# Check it to make sure they were joined correctly.
compare.head()

,County,Total Ballots Cast - Gender,Total Female,Total Male,Total Not Identified,Total Ballots Cast - Age,Age 18-29,Age 30-39,Age 40-49,Age 50-59,...,Total White (W),Total Not Identified (U),equal,Registered Voters,Total Ballots Cast - Total,Democrat Ballots,Republican Ballots,Absentee Ballots,Turnout as a Percentage of Registered Voters,Absentee as Percentage of Total Ballots
0,Autauga,28401,15750,12581,82,28401,3600,3991,4631,5223,...,22571,76,True,46291,28388,4960,13269,1670,0.6133,0.0588
1,Baldwin,122501,66932,55452,117,122501,11800,13847,17274,20669,...,112966,260,True,207643,122542,14334,69308,7771,0.5902,0.0634
2,Barbour,9901,5646,4236,19,9901,970,949,1206,1670,...,5979,40,True,17666,9919,3409,2419,593,0.5615,0.0598
3,Bibb,9283,5017,4264,2,9283,1275,1211,1323,1737,...,7947,25,True,15152,9278,1245,4628,351,0.6123,0.0378
4,Blount,28255,15026,13213,16,28255,3914,3800,4195,5005,...,27068,47,True,43601,28138,1534,18524,965,0.6454,0.0343


In [ ]:
# Compute the difference between Total Ballots Cast and Total Ballots Cast - Gender (age, and race - remember the last three have the same values)
compare["difference"] = compare["Total Ballots Cast - Total"] - compare["Total Ballots Cast - Gender"]

In [ ]:
# Compute the absolute value of the differences just to see how far off they are from each other.
compare["abs_difference"] = compare["difference"].abs()

In [ ]:
# Get the proportion of the difference.
compare["difference_percent"] = (compare["abs_difference"]/compare["Registered Voters"])

In [ ]:
compare.sort_values("difference_percent", ascending=False)[["County", "Total Ballots Cast - Gender", "Total Ballots Cast - Age", "Total Ballots Cast - Race", "Total Ballots Cast - Total", "Registered Voters", "difference_percent", "difference", "abs_difference"]].head()
# The highest difference_percent is 0.056314
# So the difference in values between Total Ballots Cast for gender, age, and race aren't too different from the total_df values for Total Ballots Cast

,County,Total Ballots Cast - Gender,Total Ballots Cast - Age,Total Ballots Cast - Race,Total Ballots Cast - Total,Registered Voters,difference_percent,difference,abs_difference
17,Conecuh,5552,5552,5552,6105,9820,0.056314,553,553
6,Butler,8112,8112,8112,8530,14530,0.028768,418,418
8,Chambers,13687,13687,13687,14380,26794,0.025864,693,693
9,Cherokee,12528,12528,12528,13030,21383,0.023477,502,502
38,Lauderdale,42188,42188,42188,43816,70858,0.022976,1628,1628


In [31]:
# The compare dataset just to get a full view of what was done here.
compare

,County,Total Ballots Cast - Gender,Total Female,Total Male,Total Not Identified,Total Ballots Cast - Age,Age 18-29,Age 30-39,Age 40-49,Age 50-59,...,Registered Voters,Total Ballots Cast - Total,Democrat Ballots,Republican Ballots,Absentee Ballots,Turnout as a Percentage of Registered Voters,Absentee as Percentage of Total Ballots,difference,abs_difference,difference_percent
0,Autauga,28401,15750,12581,82,28401,3600,3991,4631,5223,...,46291,28388,4960,13269,1670,0.6133,0.0588,-13,13,0.000281
1,Baldwin,122501,66932,55452,117,122501,11800,13847,17274,20669,...,207643,122542,14334,69308,7771,0.5902,0.0634,41,41,0.000197
2,Barbour,9901,5646,4236,19,9901,970,949,1206,1670,...,17666,9919,3409,2419,593,0.5615,0.0598,18,18,0.001019
3,Bibb,9283,5017,4264,2,9283,1275,1211,1323,1737,...,15152,9278,1245,4628,351,0.6123,0.0378,-5,5,0.000330
4,Blount,28255,15026,13213,16,28255,3914,3800,4195,5005,...,43601,28138,1534,18524,965,0.6454,0.0343,-117,117,0.002683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,Tuscaloosa,85380,48497,36790,93,85380,13505,12401,13533,14310,...,154607,85509,24077,35350,4264,0.5531,0.0499,129,129,0.000834
62,Walker,29843,16062,13749,32,29843,3726,3792,4129,5180,...,49215,29888,2777,17775,859,0.6073,0.0287,45,45,0.000914
63,Washington,8463,4605,3855,3,8463,1047,989,1125,1458,...,13788,8489,1415,4204,447,0.6157,0.0527,26,26,0.001886
64,Wilcox,5268,3028,2240,0,5268,597,561,701,929,...,8557,5287,2827,1036,693,0.6179,0.1311,19,19,0.002220
